# sanoTTS — retrain the Indonesian voice (point 1: indo-g2p ids + bigger decoder)

Full distillation run per `docs/distillation-recipe.md`, adapted for the `id` voice:

1. **Corpus**: ~4k Indonesian sentences from `wikimedia/wikipedia` (id), cleaned + filtered
2. **Front end**: indo-g2p (the same bundle the web demo uses — phonemize via a Node subprocess, stress rules identical to `web/id_g2p_map.js`)
3. **Teacher latents**: `id_ID-news_tts-medium` rendered with the indo-g2p ids injected (`voice.phonemize` monkeypatched per row)
4. **Students**: duration (tiny) → acoustic latent (token-context + adversarial) → decoder (teacher-init piperlite, larger 1.8M-class channel config) → joint fine-tune
5. **Listen**: render eval rows with teacher / oracle / student lanes

**Runtime:** T4 GPU. Budget ~4–6 h wall clock (mostly pack render + decoder stages). Free Colab survives it, but keep the tab open. Every cell is restartable — artifacts land on disk under `artifacts/`.

Honest caveats (from the recipe): 3k-row pack (vs the recipe's 8k sweet spot) trades some acoustic quality for feasible pack-render time on free Colab; raise `PACK_ROWS` if you have the patience.

In [ ]:
# @title 1. Clone repo + install deps (~3 min)
REPO_URL = "https://github.com/wafik/sanoTTS.git"  # @param {type:"string"}
BRANCH = "master"  # @param {type:"string"}

!git clone -q --depth 1 -b $BRANCH $REPO_URL /content/sanoTTS
%cd /content/sanoTTS

!pip install -q piper-tts onnxruntime-gpu soundfile scipy
!apt-get -qq install -y espeak-ng > /dev/null
import torch, shutil
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "| node", shutil.which("node"))

In [ ]:
# @title 2. Download the Indonesian Piper teacher (~60 MB)
import urllib.request
from pathlib import Path

TEACHER_DIR = Path("models/teachers/id_ID-news_tts-medium")
TEACHER_DIR.mkdir(parents=True, exist_ok=True)
BASE = "https://huggingface.co/rhasspy/piper-voices/resolve/main/id/id_ID/news_tts/medium/"
for fn in ["id_ID-news_tts-medium.onnx", "id_ID-news_tts-medium.onnx.json"]:
    dst = TEACHER_DIR / fn
    if not dst.exists():
        urllib.request.urlretrieve(BASE + fn + "?download=true", dst)
    print(dst, dst.stat().st_size, "bytes")

In [ ]:
# @title 3. Indonesian corpus from Wikipedia id (~4k clean sentences)
import json, re, urllib.request
from pathlib import Path

TARGET = 4000   # pack rows; raise to 8000 for recipe-sweet-spot quality if you accept longer renders
EVAL_COUNT = 128

def fetch_rows(limit_pages=40):
    """Page through wikimedia/wikipedia (20231101.id) via the datasets-server API (no auth)."""
    rows, offset = [], 0
    while len(rows) < limit_pages:
        url = ("https://datasets-server.huggingface.co/rows"
               "?dataset=wikimedia%2Fwikipedia&config=20231101.id&split=train"
               f"&offset={offset}&length=100")
        with urllib.request.urlopen(url) as r:
            batch = json.load(r)["rows"]
        if not batch:
            break
        rows.extend(batch)
        offset += len(batch)
    return [r["row"]["text"] for r in rows]

texts = fetch_rows()
print("fetched articles:", len(texts))

# sentence split + clean: keep 30-160 char sentences, mostly letters/spaces, dedupe
seen, sentences = set(), []
for text in texts:
    for s in re.split(r"(?<=[.!?])\s+", text):
        s = re.sub(r"\s+", " ", s).strip()
        if not (30 <= len(s) <= 160):
            continue
        letters = sum(c.isalpha() or c.isspace() for c in s)
        if letters / len(s) < 0.92:
            continue
        if s in seen:
            continue
        seen.add(s)
        sentences.append(s)
        if len(sentences) >= TARGET + EVAL_COUNT:
            break
    if len(sentences) >= TARGET + EVAL_COUNT:
        break

print("clean sentences:", len(sentences))
Path("corpus").mkdir(exist_ok=True)
eval_rows = sentences[:EVAL_COUNT]
train_rows = sentences[EVAL_COUNT:EVAL_COUNT + TARGET]
for name, rows in [("id_train.jsonl", train_rows), ("id_eval.jsonl", eval_rows)]:
    with open(f"corpus/{name}", "w", encoding="utf-8") as f:
        for i, t in enumerate(rows):
            f.write(json.dumps({"row_id": f"{'ev' if 'eval' in name else 'tr'}-{i:05d}", "text": t}, ensure_ascii=False) + "\n")
print("train", len(train_rows), "| eval", len(eval_rows))
for t in train_rows[:3]: print(" ", t)

In [ ]:
# @title 4. Phonemize the corpus with indo-g2p (Node + the web demo's own bundle)
# The repo's vendored web/id_g2p.js IS indo-g2p. We drive it from Node (preinstalled on Colab),
# applying the exact stress + g-rewrite rules of web/id_g2p_map.js, and emit per-sentence IPA.
# The pack builder then gets these IPA strings via a voice.phonemize monkeypatch — the teacher's
# phoneme_id_map is the same 161-symbol espeak inventory (ʔ ə ɛ ŋ ɡ ˈ ... verified).
import json, subprocess
from pathlib import Path

NODE_SCRIPT = r'''
const fs = require("fs"), vm = require("vm");
const ctx = { console };
vm.createContext(ctx);
for (const f of ["web/id_g2p.js", "web/id_cp_table.js", "web/id_g2p_map.js"]) {
  vm.runInContext(fs.readFileSync(f, "utf8"), ctx);
}
const rows = fs.readFileSync(process.argv[2], "utf8").trim().split("\n").map(JSON.parse);
const out = [];
for (const r of rows) {
  const { ids, skipped } = ctx.SaanoIdMap.textToIds(r.text);
  if (skipped.length > 0) continue;              // rare; drop sentences with unmappable chars
  // de-frame ids back to a phoneme-id sequence the pack builder can map: keep only non-PAD ids
  const seq = ids.filter((v, i) => i >= 2 && v !== 0 && i < ids.length - 1);
  out.push(JSON.stringify({ row_id: r.row_id, text: r.text, piper_ids: seq }));
}
fs.writeFileSync(process.argv[3], out.join("\n"));
console.log("phonemized", out.length, "of", rows.length);
'''
Path("/tmp/phonemize.js").write_text(NODE_SCRIPT, encoding="utf-8")

for src, dst in [("corpus/id_train.jsonl", "corpus/id_train_ipa.jsonl"),
                 ("corpus/id_eval.jsonl", "corpus/id_eval_ipa.jsonl")]:
    subprocess.run(["node", "/tmp/phonemize.js", src, dst], check=True)
print(open("corpus/id_train_ipa.jsonl", encoding="utf-8").readline())

In [ ]:
# @title 5. Render teacher latents with indo-g2p ids → probe packs (~2-3 h for 4k rows on T4)
# The pack builder calls voice.phonemize(text); the wrapper below patches it to return
# our precomputed indo-g2p ids (mapped back through the onnx json map) so the TEACHER
# ITSELF speaks indo-g2p phonemes. No direct import needed here — cell 5b runs the
# wrapper via CLI (it manages its own sys.path).
import json
from pathlib import Path

WRAPPER = r'''
import json, sys
from pathlib import Path
sys.path.insert(0, "tools")
import build_piper_vits_roota_probe_pack as B
from piper import PiperVoice

ids_map = {}
src = Path(sys.argv[sys.argv.index("--source-jsonl") + 1])
for line in open(src, encoding="utf-8"):
    row = json.loads(line)
    ids_map[row["text"]] = row["piper_ids"]
print("wrapper: loaded", len(ids_map), "precomputed id rows from", src)

_orig_load = PiperVoice.load
def patched_load(*a, **kw):
    voice = _orig_load(*a, **kw)
    inv = {}
    for sym, ids in voice.config.phoneme_id_map.items():
        if len(ids) == 1:
            inv.setdefault(ids[0], []).append(sym)
    def phonemize(text):
        seq = ids_map.get(text.strip())
        if seq is None:
            raise RuntimeError("no precomputed ids for: " + text[:60])
        # drop BOS/PAD/EOS framing + pads: back to phoneme strings the builder maps via phonemes_to_ids
        syms = []
        for pid in seq:
            cands = inv.get(pid)
            if not cands:
                raise RuntimeError(f"id {pid} not in map")
            syms.append(cands[0])
        return [syms]
    voice.phonemize = phonemize
    return voice
PiperVoice.load = patched_load
B.main()
'''
Path("/tmp/pack_wrapper.py").write_text(WRAPPER, encoding="utf-8")
print("wrapper ready")

In [ ]:
# @title 5b. Build train pack (4000) + eval pack (128)
!python /tmp/pack_wrapper.py \
  --model models/teachers/id_ID-news_tts-medium/id_ID-news_tts-medium.onnx \
  --config models/teachers/id_ID-news_tts-medium/id_ID-news_tts-medium.onnx.json \
  --source-jsonl corpus/id_train_ipa.jsonl \
  --out-dir packs/train4k --tensor-mode acoustic --allow-text-only-source \
  --noise-scale 0 --length-scale 1 --noise-w 0 --progress-interval 200

!python /tmp/pack_wrapper.py \
  --model models/teachers/id_ID-news_tts-medium/id_ID-news_tts-medium.onnx \
  --config models/teachers/id_ID-news_tts-medium/id_ID-news_tts-medium.onnx.json \
  --source-jsonl corpus/id_eval_ipa.jsonl \
  --out-dir packs/eval128 --tensor-mode decoder --allow-text-only-source \
  --noise-scale 0 --length-scale 1 --noise-w 0 --progress-interval 50

# decoder training also needs a small train-side pack with waveforms
!python /tmp/pack_wrapper.py \
  --model models/teachers/id_ID-news_tts-medium/id_ID-news_tts-medium.onnx \
  --config models/teachers/id_ID-news_tts-medium/id_ID-news_tts-medium.onnx.json \
  --source-jsonl corpus/id_train_ipa.jsonl --skip-rows 4000 --max-rows 512 \
  --out-dir packs/train512 --tensor-mode decoder --allow-text-only-source \
  --noise-scale 0 --length-scale 1 --noise-w 0 --progress-interval 50 || true
echo 'NOTE: if train512 came up empty (skip-rows > available), re-run with --skip-rows 3500'

In [ ]:
# @title 6. Teacher decoder CUT (ONNX oracle) (~2 min)
!python tools/extract_piper_vits_decoder_cut.py \
  --model models/teachers/id_ID-news_tts-medium/id_ID-news_tts-medium.onnx \
  --pack-dir packs/train512 --latent-channels 192 --out-dir cut/

In [ ]:
# @title 7a. Duration student (~10 min)
!python tools/train_roota_piper_duration_student.py \
  --pack-dir packs/train4k --eval-pack-dir packs/eval128 \
  --hidden 32 --depth 3 --steps 4000 --out-dir students/duration

In [ ]:
# @title 7b. Acoustic latent student (token-context, adversarial) (~1.5-2 h)
!python tools/train_roota_piper_latent_student.py \
  --pack-dir packs/train4k --architecture token_context --hidden 96 --depth 5 --token-depth 3 \
  --steps 50000 --latent-adv-weight 0.1 --latent-adv-start-step 1000 \
  --out-dir students/acoustic

In [ ]:
# @title 7c. Decoder student — teacher-init recovery stage (~1.5 h)
# 1.8M-class channel config (192,128,64,32) — the README frontier shows quality lives in decoder capacity.
!python tools/train_roota_piper_decoder_student.py \
  --pack-dir packs/train512 --teacher-decoder cut/ \
  --variant piperlite --channels 192,128,64,32 --rank-ratio 0.5 --steps 40000 \
  --out-dir students/decoder-recovery

In [ ]:
# @title 7d. Decoder z-mix stage (~1.5 h)
!python tools/train_roota_piper_decoder_student.py \
  --pack-dir packs/train512 --teacher-decoder cut/ \
  --init-decoder-checkpoint students/decoder-recovery/decoder-student.pt \
  --variant piperlite --channels 192,128,64,32 --rank-ratio 0.5 --steps 40000 \
  --acoustic-checkpoint students/acoustic/latent-student.pt --acoustic-latent-mix-prob 0.5 \
  --out-dir students/decoder-zmix

In [ ]:
# @title 7e. Joint acoustic + decoder fine-tune (~45 min)
!python tools/train_roota_joint_z_finetune.py \
  --pack-dir packs/train512 \
  --teacher-decoder cut/ \
  --acoustic-checkpoint students/acoustic/latent-student.pt \
  --decoder-checkpoint students/decoder-zmix/decoder-student.pt \
  --steps 20000 --out-dir students/joint

In [ ]:
# @title 8. Render A/B eval lanes (teacher / oracle / student) + zip everything
!python tools/render_eval_dirs.py --help | head -30

In [ ]:
# @title 9. Package checkpoints for download
import shutil
shutil.make_archive("/content/id-retrain-checkpoints", "zip", "students")
shutil.make_archive("/content/id-retrain-packs", "zip", "packs/eval128")
print("download /content/id-retrain-checkpoints.zip + /content/id-retrain-packs.zip")
print("next: local listen via tools/serve_roota_arbitrary_tts_dashboard.py (it loads these checkpoints),")
print("then packaging to a web bundle — needs the repo's exporter (owner research repo), or we port one.")